# Agilent LCR Control (Room Temperature)
Automated control script for purely room-temperature LCR measurements. Does not connect to or rely on the Janis cryostat.

In [ ]:
# Imports
from nfoinstruments.drivers import E4890A
from nfoinstruments.drivers.setup import MeasurementSetup
from nfoinstruments.procedures.utils import (
    run_temperature_bias_sweep_with_live_plot,
    run_cv_sweep_with_live_plot,
    run_time_scan_with_live_plot,
    load_and_plot_is,
    load_and_plot_cv,
    load_and_plot_time_scan
)
from pathlib import Path
import scipy.constants as const

# Define output path for saving CSVs
output_path = Path("./Data/Agilent_Only")
output_path.mkdir(parents=True, exist_ok=True)
sweep_name = "RoomTemp_Device_01"

# Initialize instruments
mm = MeasurementSetup()

# Connect to devices (update GPIB addresses if needed)
mm.connect_to_devices({
    'GPIB1::17::INSTR': E4890A
})

lcr = mm.devices['GPIB1::17::INSTR']

# Configure LCR meter
lcr.measurement_type = E4890A.MeasurementType.ZTD  # Options: ZTD (Z, θ), RX (R, X), CPD (Cp, D), CSD (Cs, D)...
lcr.measurement_time = E4890A.MeasurementTime.LONG # Options: SHORT, MEDIUM, LONG
lcr.averages = 1                                 # 1 to 256
lcr.signal_type = E4890A.SignalType.VOLTAGE        # VOLTAGE or CURRENT
lcr.alc_enabled = False                            # Automatic Level Control (True/False) - MUST BE FALSE TO PROTECT MEMRISTOR

# Initialize into Standby Mode (Protect Sample during probe landing)
print("Initializing LCR into STANDBY MODE...")
lcr.signal_amplitude = 0.0
lcr.bias = 0.0
lcr.trigger_source = 'BUS'
print("System Ready.")

## Optional: Parasitic Calibration (Open/Short)

In [ ]:
# ====================================================================
# PERFORM NEW OPEN/SHORT CORRECTIONS (Run once per setup)
# ====================================================================
perform_new_calibration = False
cable_length_m = 1

if perform_new_calibration:
    lcr.correction_length = cable_length_m
    print(f"Cable Length Correction set to: {cable_length_m}m")
    
    print("--- OPEN CORRECTION ---")
    input("1. Lift probes to measurement height in air.\n2. Press ENTER to begin...")
    lcr.perform_open_correction()
    
    print("--- SHORT CORRECTION ---")
    input("1. Land probes on shorting pad.\n2. Press ENTER to begin...")
    lcr.perform_short_correction()
    
    lcr.correction_open_enabled = True
    lcr.correction_short_enabled = True
    print("New corrections are saved in memory and ENABLED.")
else:
    print("Skipping physical calibration.")

In [ ]:
# ====================================================================
# TOGGLE EXISTING CORRECTIONS
# ====================================================================
enable_corrections = False
cable_length_m = 1

if enable_corrections:
    lcr.correction_length = cable_length_m
    lcr.correction_open_enabled = True
    lcr.correction_short_enabled = True
    print(f"Parasitic Corrections are currently: ENABLED (Cable: {cable_length_m}m).")
else:
    lcr.correction_length = 0
    lcr.correction_open_enabled = False
    lcr.correction_short_enabled = False
    print("Parasitic Corrections are currently: DISABLED (Cable: 0m).")

## Device Capacitance Reference (C_dev)
Calculate the theoretical parallel-plate capacitance of your device to use as a reference line in the Bode plots.

In [ ]:
# Example: Calculate ideal C_dev for a parallel plate capacitor
# C = (epsilon_0 * epsilon_r * Area) / thickness

epsilon_r = 10.0            # Relative permittivity of your dielectric
area_m2 = 100e-6 * 100e-6   # Area in square meters (e.g., 100um x 100um)
thickness_m = 10e-9         # Thickness in meters (e.g., 10nm)

# C_dev = (const.epsilon_0 * epsilon_r * area_m2) / thickness_m
# print(f"Calculated C_dev: {C_dev:.2e} Farads")

# For now, we will set it manually or leave as None
my_c_dev = None # Set to 1e-12 (1pF) for example


## 1. IS Scans (Impedance Spectroscopy)

In [ ]:
# Define parameters for IS Sweep
# Note: Since there is no Janis connected, we pass [295] as a dummy temperature.
temperatures = [295]  
dc_biases = [0.0, 0.5, 1.0]

# Run measurement
next_run = run_temperature_bias_sweep_with_live_plot(
    parent_dir=output_path,
    sweep_name=sweep_name,
    temp_points=temperatures,
    bias_points=dc_biases,
    janis_ctrl=None,     # NO JANIS
    lcr_ctrl=lcr,
    run_count_start=1,
    extra_settle_time=0, # No need to wait for temperature
    log_y_left=True, 
    log_y_right=False, 
    y_range_left=None,
    y_range_right=None,
    remove_outliers=False,
    C_plot=True,         # Set True to extract and plot Cp and Rp
    C_dev=my_c_dev       # Reference capacitance line
)

### Plot IS Measurements (Standalone)

In [ ]:
# Plot saved IS data
load_and_plot_is(
    parent_dir=output_path, 
    sweep_name=sweep_name,
    temp_points=temperatures,
    bias_points=dc_biases, 
    run_num=None,
    log_y_left=True, 
    log_y_right=False,
    y_range_left=None,        
    y_range_right=None,
    remove_outliers=False,
    C_plot=True,
    C_dev=my_c_dev
)

## 2. Capacitance-Voltage (CV) Scans

In [ ]:
# Define parameters for CV Sweep
Vmin = -2.0
Vmax = 2.0
Vstep = 0.1
cv_Vrms = 0.05
cv_cycles = 1

cv_freq_points = [1e4, 1e5] # e.g. 10kHz and 100kHz
temperatures = [295]

next_run = run_cv_sweep_with_live_plot(
    parent_dir=output_path,
    sweep_name="CV_" + sweep_name,
    temp_points=temperatures,
    freq_points=cv_freq_points,
    Vmin=Vmin,
    Vmax=Vmax,
    Vstep=Vstep,
    Vrms=cv_Vrms,
    cycles=cv_cycles,
    janis_ctrl=None, # NO JANIS
    lcr_ctrl=lcr,
    run_count_start=next_run,
    extra_settle_time=0,
    log_y_left=False, 
    log_y_right=False,
    y_range_left=None,
    y_range_right=None,
    remove_outliers=False,
    C_plot=False, 
    C_dev=None
)

### Plot CV Measurements (Standalone)

In [ ]:
load_and_plot_cv(
    parent_dir=output_path, 
    sweep_name="CV_" + sweep_name,
    temp_points=temperatures,
    freq_points=cv_freq_points,
    run_num=None, 
    log_y_left=False, 
    log_y_right=False,
    y_range_left=None,
    y_range_right=None,
    remove_outliers=False,
    C_plot=False,
    C_dev=None
)

## 3. Time Scan (Drift / Electroforming)

In [ ]:
# Define parameters for Time Scan
scan_duration = 600   # Seconds to monitor
update_interval = 5   # Seconds between live plot updates

ts_temps = [295]      # Dummy room temp
ts_freqs = [1e4]      # Hz
ts_Vrms = [0.05]      # V
ts_Vdc = [1.0]        # V

ts_Z_threshold = None # e.g. 1000 for 1kOhm software stop

next_run = run_time_scan_with_live_plot(
    parent_dir=output_path,
    sweep_name="TimeScan_" + sweep_name,
    temp_points=ts_temps,
    freq_points=ts_freqs,
    Vdc_points=ts_Vdc,
    Vrms_points=ts_Vrms,
    scan_duration=scan_duration,
    janis_ctrl=None, # NO JANIS
    lcr_ctrl=lcr,
    update_interval=update_interval,
    extra_settle_time=0,
    run_count_start=next_run,
    log_y_left=False, 
    log_y_right=False,
    y_range_left=None,
    y_range_right=None,
    remove_outliers=False,
    Z_threshold=ts_Z_threshold,
    C_plot=True,
    C_dev=my_c_dev
)

### Plot Time Scan (Standalone)

In [ ]:
load_and_plot_time_scan(
    parent_dir=output_path, 
    sweep_name="TimeScan_" + sweep_name,
    freq_points=ts_freqs, 
    Vdc_points=ts_Vdc, 
    Vrms_points=ts_Vrms, 
    run_num=None,
    log_y_left=False, 
    log_y_right=False,
    y_range_left=None,
    y_range_right=None,
    remove_outliers=False,
    C_plot=True,
    C_dev=my_c_dev
)